In [ ]:
!pip install -q -U "duckdb>=1.4.1,<1.6" pandas openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 100.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


In [ ]:
import duckdb
import pandas as pd
import os
from IPython.display import display

print("DuckDB version:")
print(duckdb.__version__)

# MotherDuck (wilayah Tokyo) hanya mendukung klien DuckDB 1.4.1 - 1.5.6
versi = tuple(int(x) for x in duckdb.__version__.split(".")[:3])
assert versi >= (1, 4, 1), (
    f"DuckDB {duckdb.__version__} terlalu lama untuk MotherDuck. "
    "Pilih menu Runtime > Restart session, lalu jalankan ulang dari sel paling atas."
)

DuckDB version:
1.5.6


In [ ]:
FILE_IPM = "/content/IPM 2025.xlsx"

FILE_PENDIDIKAN = "/content/dataset_integrasi_sekolah_guru_murid_jawa_timur.xlsx"

FILE_RLS = "/content/rataratalamasekolah.xlsx"

In [ ]:
df_ipm = pd.read_excel(
    FILE_IPM,
    sheet_name="IPM Jawa Timur"
)

print("Shape:", df_ipm.shape)
display(df_ipm.head())

Shape: (38, 2)


,Kabupaten/Kota,IPM
0,Pacitan,72.26
1,Ponorogo,74.65
2,Trenggalek,73.39
3,Tulungagung,75.88
4,Blitar,74.43


In [ ]:
df_ipm.columns = ["kabupaten_kota", "ipm"]

df_ipm["kabupaten_kota"] = (
    df_ipm["kabupaten_kota"]
    .astype(str)
    .str.strip()
)

df_ipm["ipm"] = pd.to_numeric(
    df_ipm["ipm"],
    errors="coerce"
)

df_ipm["tahun"] = 2025

display(df_ipm.head())

,kabupaten_kota,ipm,tahun
0,Pacitan,72.26,2025
1,Ponorogo,74.65,2025
2,Trenggalek,73.39,2025
3,Tulungagung,75.88,2025
4,Blitar,74.43,2025


In [ ]:
df_sekolah = pd.read_excel(
    FILE_PENDIDIKAN,
    sheet_name="Jumlah Sekolah"
)

display(df_sekolah.head())

,Kabupaten/Kota,SD,SMP,SMA
0,Bangkalan,679,263,67
1,Banyuwangi,808,230,51
2,Blitar,664,105,23
3,Bojonegoro,711,110,50
4,Bondowoso,466,114,29


In [ ]:
df_sekolah.columns = [
    "kabupaten_kota",
    "sd",
    "smp",
    "sma"
]

df_sekolah["kabupaten_kota"] = (
    df_sekolah["kabupaten_kota"]
    .astype(str)
    .str.strip()
)

for col in ["sd", "smp", "sma"]:
    df_sekolah[col] = pd.to_numeric(
        df_sekolah[col],
        errors="coerce"
    )

df_sekolah["tahun"] = 2025

display(df_sekolah.head())

,kabupaten_kota,sd,smp,sma,tahun
0,Bangkalan,679,263,67,2025
1,Banyuwangi,808,230,51,2025
2,Blitar,664,105,23,2025
3,Bojonegoro,711,110,50,2025
4,Bondowoso,466,114,29,2025


In [ ]:
df_guru = pd.read_excel(
    FILE_PENDIDIKAN,
    sheet_name="Jumlah Guru"
)

df_guru.columns = [
    "kabupaten_kota",
    "sd",
    "smp",
    "sma"
]

df_guru["kabupaten_kota"] = (
    df_guru["kabupaten_kota"]
    .astype(str)
    .str.strip()
)

for col in ["sd", "smp", "sma"]:
    df_guru[col] = pd.to_numeric(
        df_guru[col],
        errors="coerce"
    )

df_guru["tahun"] = 2025

display(df_guru.head())

,kabupaten_kota,sd,smp,sma,tahun
0,Bangkalan,5493,2871,1043,2025
1,Banyuwangi,6665,3376,1314,2025
2,Blitar,5834,2113,540,2025
3,Bojonegoro,6343,2017,1119,2025
4,Bondowoso,4733,1573,539,2025


In [ ]:
df_murid = pd.read_excel(
    FILE_PENDIDIKAN,
    sheet_name="Jumlah Murid"
)

df_murid.columns = [
    "kabupaten_kota",
    "sd",
    "smp",
    "sma"
]

df_murid["kabupaten_kota"] = (
    df_murid["kabupaten_kota"]
    .astype(str)
    .str.strip()
)

for col in ["sd", "smp", "sma"]:
    df_murid[col] = pd.to_numeric(
        df_murid[col],
        errors="coerce"
    )

df_murid["tahun"] = 2025

display(df_murid.head())

,kabupaten_kota,sd,smp,sma,tahun
0,Bangkalan,99280,37499,17179,2025
1,Banyuwangi,101718,52277,19886,2025
2,Blitar,60476,33253,9115,2025
3,Bojonegoro,65664,32074,17904,2025
4,Bondowoso,49880,16099,7424,2025


In [ ]:
df_rls_raw = pd.read_excel(
    FILE_RLS,
    sheet_name="Sheet1",
    header=None
)

display(df_rls_raw.head(10))

,0,1
0,Kab/Kota,NaN
1,NaN,Rata-rata Lama Sekolah (Penduduk 15 tahun keat...
2,NaN,2025
3,Pacitan,8.26
4,Ponorogo,8.35
5,Trenggalek,8.47
6,Tulungagung,8.98
7,Blitar,8.56
8,Kediri,8.75
9,Malang,8.56


In [ ]:
# ==========================================
# MEMBERSIHKAN DATA RATA-RATA LAMA SEKOLAH
# ==========================================

df_rls_raw = pd.read_excel(
    FILE_RLS,
    sheet_name=0,
    header=None
)

# Ambil dua kolom pertama
df_rls = df_rls_raw.iloc[:, :2].copy()

df_rls.columns = [
    "kabupaten_kota",
    "rata_lama_sekolah"
]

# Bersihkan nama wilayah
df_rls["kabupaten_kota"] = (
    df_rls["kabupaten_kota"]
    .astype("string")
    .str.strip()
)

# Konversi RLS menjadi numerik
df_rls["rata_lama_sekolah"] = pd.to_numeric(
    df_rls["rata_lama_sekolah"],
    errors="coerce"
)

# Buang:
# 1. Baris kosong
# 2. Header "Kab/Kota"
# 3. Baris agregat "Jawa Timur"
# 4. Baris yang tidak memiliki nilai RLS

df_rls = df_rls[
    df_rls["kabupaten_kota"].notna()
    & (df_rls["kabupaten_kota"] != "Kab/Kota")
    & (df_rls["kabupaten_kota"] != "Jawa Timur")
    & df_rls["rata_lama_sekolah"].notna()
].copy()

# Tambahkan tahun
df_rls["tahun"] = 2025

# Reset index
df_rls = df_rls.sort_values(
    "kabupaten_kota"
).reset_index(drop=True)

# Tampilkan
display(df_rls)

print("Jumlah wilayah RLS:", len(df_rls))

,kabupaten_kota,rata_lama_sekolah,tahun
0,Bangkalan,7.12,2025
1,Banyuwangi,8.28,2025
2,Blitar,8.56,2025
3,Bojonegoro,8.36,2025
4,Bondowoso,7.01,2025
5,Gresik,10.35,2025
6,Jember,7.29,2025
7,Jombang,9.23,2025
8,Kediri,8.75,2025
9,Kota Batu,10.18,2025


Jumlah wilayah RLS: 38


In [ ]:
print("IPM       :", len(df_ipm))
print("Sekolah   :", len(df_sekolah))
print("Guru      :", len(df_guru))
print("Murid     :", len(df_murid))
print("RLS       :", len(df_rls))

IPM       : 38
Sekolah   : 38
Guru      : 38
Murid     : 38
RLS       : 38


In [ ]:
wilayah_ipm = set(df_ipm["kabupaten_kota"])
wilayah_sekolah = set(df_sekolah["kabupaten_kota"])
wilayah_guru = set(df_guru["kabupaten_kota"])
wilayah_murid = set(df_murid["kabupaten_kota"])
wilayah_rls = set(df_rls["kabupaten_kota"])

semua_wilayah = (
    wilayah_ipm
    | wilayah_sekolah
    | wilayah_guru
    | wilayah_murid
    | wilayah_rls
)

print("Total wilayah unik:", len(semua_wilayah))

Total wilayah unik: 38


**Pembuatan DuckDB**

In [ ]:
con = duckdb.connect(
    "/content/jawa_timur.duckdb"
)

print(
    con.execute("SELECT version()").fetchone()[0]
)

v1.5.6


raw

In [ ]:
con.execute("CREATE SCHEMA IF NOT EXISTS raw")

In [ ]:
con.register("tmp_ipm", df_ipm)
con.register("tmp_sekolah", df_sekolah)
con.register("tmp_guru", df_guru)
con.register("tmp_murid", df_murid)
con.register("tmp_rls", df_rls)

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE raw.ipm AS
SELECT * FROM tmp_ipm
""")

con.execute("""
CREATE OR REPLACE TABLE raw.jumlah_sekolah AS
SELECT * FROM tmp_sekolah
""")

con.execute("""
CREATE OR REPLACE TABLE raw.jumlah_guru AS
SELECT * FROM tmp_guru
""")

con.execute("""
CREATE OR REPLACE TABLE raw.jumlah_murid AS
SELECT * FROM tmp_murid
""")

con.execute("""
CREATE OR REPLACE TABLE raw.rata_lama_sekolah AS
SELECT * FROM tmp_rls
""")

In [ ]:
con.sql("""
SELECT table_name
FROM information_schema.tables
WHERE table_schema = 'raw'
ORDER BY table_name
""").show()

┌───────────────────┐
│    table_name     │
│      varchar      │
├───────────────────┤
│ ipm               │
│ jumlah_guru       │
│ jumlah_murid      │
│ jumlah_sekolah    │
│ rata_lama_sekolah │
└───────────────────┘



In [ ]:
con.execute("""
CREATE SCHEMA IF NOT EXISTS dw
""")

print("Schema DW berhasil dibuat.")

Schema DW berhasil dibuat.


In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dw.dim_wilayah AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY kabupaten_kota
    ) AS wilayah_id,
    kabupaten_kota
FROM (
    SELECT DISTINCT kabupaten_kota
    FROM raw.ipm
)
ORDER BY kabupaten_kota
""")

In [ ]:
con.sql("""
SELECT *
FROM dw.dim_wilayah
ORDER BY wilayah_id
""").show()

┌────────────┬────────────────┐
│ wilayah_id │ kabupaten_kota │
│   int64    │    varchar     │
├────────────┼────────────────┤
│          1 │ Bangkalan      │
│          2 │ Banyuwangi     │
│          3 │ Blitar         │
│          4 │ Bojonegoro     │
│          5 │ Bondowoso      │
│          6 │ Gresik         │
│          7 │ Jember         │
│          8 │ Jombang        │
│          9 │ Kediri         │
│         10 │ Kota Batu      │
│          · │    ·           │
│          · │    ·           │
│          · │    ·           │
│         29 │ Pasuruan       │
│         30 │ Ponorogo       │
│         31 │ Probolinggo    │
│         32 │ Sampang        │
│         33 │ Sidoarjo       │
│         34 │ Situbondo      │
│         35 │ Sumenep        │
│         36 │ Trenggalek     │
│         37 │ Tuban          │
│         38 │ Tulungagung    │
└────────────┴────────────────┘
  38 rows (20 shown)2 columns



In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dw.dim_tahun AS
SELECT
    ROW_NUMBER() OVER (
        ORDER BY tahun
    ) AS tahun_id,
    tahun
FROM (
    SELECT DISTINCT tahun
    FROM raw.ipm
)
""")

In [ ]:
con.sql("""
SELECT *
FROM dw.dim_tahun
""").show()

┌──────────┬───────┐
│ tahun_id │ tahun │
│  int64   │ int64 │
├──────────┼───────┤
│        1 │  2025 │
└──────────┴───────┘



In [ ]:
con.execute("""
CREATE OR REPLACE TABLE dw.fact_indikator_pendidikan AS

SELECT
    w.wilayah_id,
    t.tahun_id,

    i.ipm,

    s.sd AS sekolah_sd,
    s.smp AS sekolah_smp,
    s.sma AS sekolah_sma,

    g.sd AS guru_sd,
    g.smp AS guru_smp,
    g.sma AS guru_sma,

    m.sd AS murid_sd,
    m.smp AS murid_smp,
    m.sma AS murid_sma,

    r.rata_lama_sekolah

FROM raw.ipm AS i

LEFT JOIN raw.jumlah_sekolah AS s
    ON i.kabupaten_kota = s.kabupaten_kota
    AND i.tahun = s.tahun

LEFT JOIN raw.jumlah_guru AS g
    ON i.kabupaten_kota = g.kabupaten_kota
    AND i.tahun = g.tahun

LEFT JOIN raw.jumlah_murid AS m
    ON i.kabupaten_kota = m.kabupaten_kota
    AND i.tahun = m.tahun

LEFT JOIN raw.rata_lama_sekolah AS r
    ON i.kabupaten_kota = r.kabupaten_kota
    AND i.tahun = r.tahun

LEFT JOIN dw.dim_wilayah AS w
    ON i.kabupaten_kota = w.kabupaten_kota

LEFT JOIN dw.dim_tahun AS t
    ON i.tahun = t.tahun
""")

print("Fact table berhasil dibuat.")

Fact table berhasil dibuat.


In [ ]:
con.sql("""
SELECT COUNT(*) AS jumlah_data
FROM dw.fact_indikator_pendidikan
""").show()

┌─────────────┐
│ jumlah_data │
│    int64    │
├─────────────┤
│          38 │
└─────────────┘



In [ ]:
con.sql("""
SELECT
    COUNT(*) AS total_data,

    COUNT(*) FILTER (
        WHERE ipm IS NULL
    ) AS ipm_null,

    COUNT(*) FILTER (
        WHERE sekolah_sd IS NULL
    ) AS sekolah_null,

    COUNT(*) FILTER (
        WHERE guru_sd IS NULL
    ) AS guru_null,

    COUNT(*) FILTER (
        WHERE murid_sd IS NULL
    ) AS murid_null,

    COUNT(*) FILTER (
        WHERE rata_lama_sekolah IS NULL
    ) AS rls_null

FROM dw.fact_indikator_pendidikan
""").show()

┌────────────┬──────────┬──────────────┬───────────┬────────────┬──────────┐
│ total_data │ ipm_null │ sekolah_null │ guru_null │ murid_null │ rls_null │
│   int64    │  int64   │    int64     │   int64   │   int64    │  int64   │
├────────────┼──────────┼──────────────┼───────────┼────────────┼──────────┤
│         38 │        0 │            0 │         0 │          0 │        0 │
└────────────┴──────────┴──────────────┴───────────┴────────────┴──────────┘



In [ ]:
con.sql("""
SELECT
    w.kabupaten_kota,
    t.tahun,

    f.ipm,

    f.sekolah_sd,
    f.sekolah_smp,
    f.sekolah_sma,

    f.guru_sd,
    f.guru_smp,
    f.guru_sma,

    f.murid_sd,
    f.murid_smp,
    f.murid_sma,

    f.rata_lama_sekolah

FROM dw.fact_indikator_pendidikan AS f

LEFT JOIN dw.dim_wilayah AS w
    ON f.wilayah_id = w.wilayah_id

LEFT JOIN dw.dim_tahun AS t
    ON f.tahun_id = t.tahun_id

ORDER BY w.kabupaten_kota
""").show()

┌────────────────┬───────┬────────┬────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬───────────┬───────────┬───────────────────┐
│ kabupaten_kota │ tahun │  ipm   │ sekolah_sd │ sekolah_smp │ sekolah_sma │ guru_sd │ guru_smp │ guru_sma │ murid_sd │ murid_smp │ murid_sma │ rata_lama_sekolah │
│    varchar     │ int64 │ double │   int64    │    int64    │    int64    │  int64  │  int64   │  int64   │  int64   │   int64   │   int64   │      double       │
├────────────────┼───────┼────────┼────────────┼─────────────┼─────────────┼─────────┼──────────┼──────────┼──────────┼───────────┼───────────┼───────────────────┤
│ Bangkalan      │  2025 │  68.15 │        679 │         263 │          67 │    5493 │     2871 │     1043 │    99280 │     37499 │     17179 │              7.12 │
│ Banyuwangi     │  2025 │  75.17 │        808 │         230 │          51 │    6665 │     3376 │     1314 │   101718 │     52277 │     19886 │              8.28 │
│ Blitar        

In [ ]:
con.execute("""
CREATE OR REPLACE VIEW dw.v_indikator_jawa_timur AS

SELECT
    w.kabupaten_kota,
    t.tahun,

    f.ipm,

    f.sekolah_sd,
    f.sekolah_smp,
    f.sekolah_sma,

    (
        f.sekolah_sd +
        f.sekolah_smp +
        f.sekolah_sma
    ) AS total_sekolah,

    f.guru_sd,
    f.guru_smp,
    f.guru_sma,

    (
        f.guru_sd +
        f.guru_smp +
        f.guru_sma
    ) AS total_guru,

    f.murid_sd,
    f.murid_smp,
    f.murid_sma,

    (
        f.murid_sd +
        f.murid_smp +
        f.murid_sma
    ) AS total_murid,

    f.rata_lama_sekolah

FROM dw.fact_indikator_pendidikan AS f

LEFT JOIN dw.dim_wilayah AS w
    ON f.wilayah_id = w.wilayah_id

LEFT JOIN dw.dim_tahun AS t
    ON f.tahun_id = t.tahun_id
""")

print("View berhasil dibuat.")

View berhasil dibuat.


In [ ]:
df_integrasi = con.sql("""
SELECT *
FROM dw.v_indikator_jawa_timur
ORDER BY kabupaten_kota
""").df()

display(df_integrasi)

,kabupaten_kota,tahun,ipm,sekolah_sd,sekolah_smp,sekolah_sma,total_sekolah,guru_sd,guru_smp,guru_sma,total_guru,murid_sd,murid_smp,murid_sma,total_murid,rata_lama_sekolah
0,Bangkalan,2025,68.15,679,263,67,1009,5493,2871,1043,9407,99280,37499,17179,153958,7.12
1,Banyuwangi,2025,75.17,808,230,51,1089,6665,3376,1314,11355,101718,52277,19886,173881,8.28
2,Blitar,2025,74.43,664,105,23,792,5834,2113,540,8487,60476,33253,9115,102844,8.56
3,Bojonegoro,2025,73.74,711,110,50,871,6343,2017,1119,9479,65664,32074,17904,115642,8.36
4,Bondowoso,2025,71.72,466,114,29,609,4733,1573,539,6845,49880,16099,7424,73403,7.01
5,Gresik,2025,79.69,466,127,52,645,5132,2161,1297,8590,75428,36186,20909,132523,10.35
6,Jember,2025,71.57,1051,347,62,1460,8567,4514,1559,14640,162693,71292,26113,260098,7.29
7,Jombang,2025,76.37,522,137,46,705,4108,2404,1203,7715,67645,42903,19406,129954,9.23
8,Kediri,2025,76.09,666,130,27,823,6343,2599,995,9937,89933,51279,17913,159125,8.75
9,Kota Batu,2025,80.35,78,31,11,120,987,590,301,1878,15295,9262,4977,29534,10.18


In [ ]:
con.execute("""
INSTALL ducklake;
""")

con.execute("""
LOAD ducklake;
""")

print("DuckLake extension berhasil diaktifkan.")

DuckLake extension berhasil diaktifkan.


In [ ]:
# Katalog DuckLake lama dibuat dengan DuckDB 1.3.2. Hapus supaya dibuat ulang dengan versi baru.
# Aman: seluruh isinya dibuat ulang oleh sel-sel di bawah dari data Excel. Jalankan hanya saat "Run all" dari atas.
import shutil

for f in ["/content/jawa_timur.ducklake", "/content/jawa_timur.ducklake.wal"]:
    if os.path.exists(f):
        os.remove(f)
shutil.rmtree("/content/jawa_timur.ducklake.files", ignore_errors=True)

print("Katalog DuckLake lama dibersihkan.")

Katalog DuckLake lama dibersihkan.


In [ ]:
con.execute("""
ATTACH 'ducklake:/content/jawa_timur.ducklake'
AS lake_jatim;
""")

In [ ]:
con.sql("""
SELECT *
FROM duckdb_databases()
""").show()

┌───────────────┬──────────────┬──────────────────────────────┬─────────┬─────────────────────────────────────────────────┬──────────┬──────────┬──────────┬───────────┬─────────┬───────────────────────┐
│ database_name │ database_oid │             path             │ comment │                      tags                       │ internal │   type   │ readonly │ encrypted │ cipher  │        options        │
│    varchar    │    int64     │           varchar            │ varchar │              map(varchar, varchar)              │ boolean  │ varchar  │ boolean  │  boolean  │ varchar │ map(varchar, varchar) │
├───────────────┼──────────────┼──────────────────────────────┼─────────┼─────────────────────────────────────────────────┼──────────┼──────────┼──────────┼───────────┼─────────┼───────────────────────┤
│ lake_jatim    │        22413 │ /content/jawa_timur.ducklake │ NULL    │ {data_path=/content/jawa_timur.ducklake.files/} │ false    │ ducklake │ false    │ false     │ NULL    │ {busy_tim

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE lake_jatim.fact_indikator_pendidikan AS
SELECT *
FROM dw.v_indikator_jawa_timur;
""")

In [ ]:
con.sql("""
SELECT *
FROM lake_jatim.fact_indikator_pendidikan
LIMIT 10
""").show()

┌────────────────┬───────┬────────┬────────────┬─────────────┬─────────────┬───────────────┬─────────┬──────────┬──────────┬────────────┬──────────┬───────────┬───────────┬─────────────┬───────────────────┐
│ kabupaten_kota │ tahun │  ipm   │ sekolah_sd │ sekolah_smp │ sekolah_sma │ total_sekolah │ guru_sd │ guru_smp │ guru_sma │ total_guru │ murid_sd │ murid_smp │ murid_sma │ total_murid │ rata_lama_sekolah │
│    varchar     │ int64 │ double │   int64    │    int64    │    int64    │     int64     │  int64  │  int64   │  int64   │   int64    │  int64   │   int64   │   int64   │    int64    │      double       │
├────────────────┼───────┼────────┼────────────┼─────────────┼─────────────┼───────────────┼─────────┼──────────┼──────────┼────────────┼──────────┼───────────┼───────────┼─────────────┼───────────────────┤
│ Bangkalan      │  2025 │  68.15 │        679 │         263 │          67 │          1009 │    5493 │     2871 │     1043 │       9407 │    99280 │     37499 │     17179 │

In [ ]:
con.sql("""
SELECT COUNT(*) AS jumlah_data
FROM lake_jatim.fact_indikator_pendidikan
""").show()

┌─────────────┐
│ jumlah_data │
│    int64    │
├─────────────┤
│          38 │
└─────────────┘



In [ ]:
con.execute("""
CREATE OR REPLACE TABLE lake_jatim.dim_wilayah AS
SELECT *
FROM dw.dim_wilayah;
""")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE lake_jatim.dim_tahun AS
SELECT *
FROM dw.dim_tahun;
""")

In [ ]:
con.sql("""
SELECT
    table_schema,
    table_name
FROM information_schema.tables
WHERE table_catalog = 'lake_jatim'
ORDER BY table_schema, table_name
""").show()

┌──────────────┬───────────────────────────┐
│ table_schema │        table_name         │
│   varchar    │          varchar          │
├──────────────┼───────────────────────────┤
│ main         │ dim_tahun                 │
│ main         │ dim_wilayah               │
│ main         │ fact_indikator_pendidikan │
└──────────────┴───────────────────────────┘



In [ ]:
# Top 10 IPM
con.sql("""
SELECT
    kabupaten_kota,
    ipm,
    rata_lama_sekolah,
    total_sekolah,
    total_guru,
    total_murid

FROM lake_jatim.fact_indikator_pendidikan

ORDER BY ipm DESC

LIMIT 10
""").show()

┌────────────────┬────────┬───────────────────┬───────────────┬────────────┬─────────────┐
│ kabupaten_kota │  ipm   │ rata_lama_sekolah │ total_sekolah │ total_guru │ total_murid │
│    varchar     │ double │      double       │     int64     │   int64    │    int64    │
├────────────────┼────────┼───────────────────┼───────────────┼────────────┼─────────────┤
│ Kota Surabaya  │  85.65 │             11.46 │          1106 │      20999 │      372628 │
│ Kota Malang    │  85.55 │             11.51 │           452 │       7664 │      125218 │
│ Kota Madiun    │  85.12 │             11.66 │           102 │       1776 │       28691 │
│ Sidoarjo       │  83.35 │             11.09 │           847 │      14532 │      251479 │
│ Kota Kediri    │  82.71 │              11.1 │           198 │       3224 │       50481 │
│ Kota Mojokerto │  82.35 │             11.38 │            84 │       1525 │       24148 │
│ Kota Blitar    │  82.03 │             10.76 │            93 │       1605 │       25706 │

In [ ]:
# Total Pendidikan Jawa Timur
con.sql("""
SELECT
    SUM(total_sekolah) AS total_sekolah,
    SUM(total_guru) AS total_guru,
    SUM(total_murid) AS total_murid,
    AVG(ipm) AS rata_rata_ipm,
    AVG(rata_lama_sekolah) AS rata_rata_rls

FROM lake_jatim.fact_indikator_pendidikan
""").show()

┌───────────────┬────────────┬─────────────┬───────────────────┬───────────────────┐
│ total_sekolah │ total_guru │ total_murid │   rata_rata_ipm   │   rata_rata_rls   │
│    int128     │   int128   │   int128    │      double       │      double       │
├───────────────┼────────────┼─────────────┼───────────────────┼───────────────────┤
│         25521 │     294886 │     4282023 │ 76.06447368421054 │ 8.941315789473686 │
└───────────────┴────────────┴─────────────┴───────────────────┴───────────────────┘



In [ ]:
# Korelasi IPM dan Rata Rata Lama Sekolah
con.sql("""
SELECT
    CORR(ipm, rata_lama_sekolah) AS korelasi_ipm_rls
FROM lake_jatim.fact_indikator_pendidikan
""").show()

┌────────────────────┐
│  korelasi_ipm_rls  │
│       double       │
├────────────────────┤
│ 0.9796343074904503 │
└────────────────────┘



In [ ]:
!ls -lah /content/

total 132K
drwxr-xr-x 1 root root 4.0K Oct 12 13:17  .
drwxr-xr-x 1 root root 4.0K Oct 12 13:06  ..
drwxr-xr-x 4 root root 4.0K Oct  2 13:50  .config
-rw-r--r-- 1 root root 9.3K Oct 12 13:16  dataset_integrasi_sekolah_guru_murid_jawa_timur.xlsx
-rw-r--r-- 1 root root 9.9K Oct 12 13:16 'IPM 2025.xlsx'
-rw-r--r-- 1 root root  12K Oct 12 13:17  jawa_timur.duckdb
-rw-r--r-- 1 root root  17K Oct 12 13:17  jawa_timur.duckdb.wal
-rw-r--r-- 1 root root  12K Oct 12 13:17  jawa_timur.ducklake
drwxr-xr-x 3 root root 4.0K Oct 12 13:17  jawa_timur.ducklake.files
-rw-r--r-- 1 root root  21K Oct 12 13:17  jawa_timur.ducklake.wal
-rw-r--r-- 1 root root  20K Oct 12 13:16  rataratalamasekolah.xlsx
drwxr-xr-x 1 root root 4.0K Oct  2 13:50  sample_data


In [ ]:
# File Parquet DuckLake tersimpan di subfolder main/<nama_tabel>/, jadi kedalaman perlu ditambah
!find /content/jawa_timur.ducklake.files -type f | head -30

/content/jawa_timur.ducklake.files/main/dim_wilayah/ducklake-01a13042-bae1-7dc1-96e8-4aad4f0c55e8.parquet
/content/jawa_timur.ducklake.files/main/fact_indikator_pendidikan/ducklake-01a13042-ba64-714b-96b0-060e0f6e13b2.parquet


**Pembuatan MotherDuck**

**Langkah persiapan MotherDuck**

1. Daftar/login di https://app.motherduck.com
2. Buka *Settings → Access Tokens* → buat token baru, salin.
3. Di Colab klik ikon **kunci (Secrets)** di sidebar kiri → *Add new secret* dengan nama `MOTHERDUCK_TOKEN`, tempel token, aktifkan *Notebook access*.

Token tidak ditulis langsung di kode supaya tidak ikut tersebar saat notebook dibagikan.

In [ ]:
import os, re
from google.colab import userdata

raw = userdata.get("MOTHERDUCK_TOKEN")

# Bersihkan spasi/newline tersembunyi dan tanda kutip yang ikut tertempel
token = re.sub(r"\s+", "", raw).strip('"').strip("'")

# MotherDuck membaca variabel lingkungan huruf kecil ini
os.environ["motherduck_token"] = token
os.environ["MOTHERDUCK_TOKEN"] = token

# Diagnostik (isi token tidak ditampilkan)
print("Panjang token          :", len(token))
print("Diawali 'eyJ' (JWT)    :", token.startswith("eyJ"))
print("Jumlah titik '.'       :", token.count("."), "(token normal = 2)")
print("Berubah setelah dibersihkan:", raw != token)

Panjang token          : 440
Diawali 'eyJ' (JWT)    : True
Jumlah titik '.'       : 2 (token normal = 2)
Berubah setelah dibersihkan: False


In [ ]:
# Uji token ke MotherDuck lewat dua jalur supaya penyebab error terlihat jelas
hasil = {}
for nama, dsn in [
    ("variabel lingkungan", "md:"),
    ("connection string", f"md:?motherduck_token={token}"),
]:
    try:
        c = duckdb.connect(dsn)
        c.close()
        hasil[nama] = "OK"
    except Exception as e:
        hasil[nama] = "GAGAL -> " + str(e).split("Request failed:")[-1].strip()[:140]

for nama, status in hasil.items():
    print(f"{nama:<22}: {status}")

variabel lingkungan   : OK
connection string     : OK


In [ ]:
# Buat database di MotherDuck (dengan koneksi terpisah ke cloud)
md_con = duckdb.connect("md:")

md_con.execute("CREATE DATABASE IF NOT EXISTS jatim_dw")

md_con.sql("SHOW DATABASES").show()

md_con.close()

┌───────────────────────┐
│     database_name     │
│        varchar        │
├───────────────────────┤
│ jatim_dw              │
│ md_information_schema │
│ my_db                 │
│ sample_data           │
└───────────────────────┘



In [ ]:
# Hubungkan database MotherDuck ke koneksi lokal yang sudah memuat DuckLake (hybrid query)
con.execute("ATTACH 'md:jatim_dw'")

con.sql("""
SELECT database_name, type, path
FROM duckdb_databases()
WHERE NOT internal
""").show()

┌───────────────────────┬─────────────────┬──────────────────────────────┐
│     database_name     │      type       │             path             │
│        varchar        │     varchar     │           varchar            │
├───────────────────────┼─────────────────┼──────────────────────────────┤
│ jatim_dw              │ motherduck      │ jatim_dw                     │
│ md_information_schema │ motherduck_info │ NULL                         │
│ lake_jatim            │ ducklake        │ /content/jawa_timur.ducklake │
│ jawa_timur            │ duckdb          │ /content/jawa_timur.duckdb   │
└───────────────────────┴─────────────────┴──────────────────────────────┘



In [ ]:
# Salin dimensi dan fakta dari DuckLake (lokal) ke MotherDuck (cloud)
tabel_publikasi = [
    "dim_wilayah",
    "dim_tahun",
    "fact_indikator_pendidikan",
]

for tabel in tabel_publikasi:
    con.execute(f"""
    CREATE OR REPLACE TABLE jatim_dw.main.{tabel} AS
    SELECT * FROM lake_jatim.{tabel}
    """)
    print(f"{tabel:<28} berhasil diunggah ke MotherDuck")

dim_wilayah                  berhasil diunggah ke MotherDuck
dim_tahun                    berhasil diunggah ke MotherDuck
fact_indikator_pendidikan    berhasil diunggah ke MotherDuck


In [ ]:
# Validasi: jumlah baris lokal (DuckLake) vs cloud (MotherDuck) harus sama
validasi = []

for tabel in tabel_publikasi:
    n_lokal = con.execute(f"SELECT COUNT(*) FROM lake_jatim.{tabel}").fetchone()[0]
    n_cloud = con.execute(f"SELECT COUNT(*) FROM jatim_dw.main.{tabel}").fetchone()[0]
    validasi.append({
        "tabel": tabel,
        "baris_ducklake": n_lokal,
        "baris_motherduck": n_cloud,
        "status": "OK" if n_lokal == n_cloud else "BERBEDA",
    })

display(pd.DataFrame(validasi))

,tabel,baris_ducklake,baris_motherduck,status
0,dim_wilayah,38,38,OK
1,dim_tahun,1,1,OK
2,fact_indikator_pendidikan,38,38,OK


In [ ]:
# Validasi isi: agregat di MotherDuck harus identik dengan hasil analisis sebelumnya
con.sql("""
SELECT
    SUM(total_sekolah)              AS total_sekolah,
    SUM(total_guru)                 AS total_guru,
    SUM(total_murid)                AS total_murid,
    ROUND(AVG(ipm), 4)              AS rata_rata_ipm,
    ROUND(AVG(rata_lama_sekolah), 4) AS rata_rata_rls,
    ROUND(CORR(ipm, rata_lama_sekolah), 4) AS korelasi_ipm_rls
FROM jatim_dw.main.fact_indikator_pendidikan
""").show()

┌───────────────┬────────────┬─────────────┬───────────────┬───────────────┬──────────────────┐
│ total_sekolah │ total_guru │ total_murid │ rata_rata_ipm │ rata_rata_rls │ korelasi_ipm_rls │
│    int128     │   int128   │   int128    │    double     │    double     │      double      │
├───────────────┼────────────┼─────────────┼───────────────┼───────────────┼──────────────────┤
│         25521 │     294886 │     4282023 │       76.0645 │        8.9413 │           0.9796 │
└───────────────┴────────────┴─────────────┴───────────────┴───────────────┴──────────────────┘



In [ ]:
# Uji query langsung ke cloud (koneksi baru, tanpa data lokal) — sama seperti yang dilakukan Streamlit
uji = duckdb.connect("md:jatim_dw")

uji.sql("""
SELECT
    CASE WHEN kabupaten_kota LIKE 'Kota %' THEN 'Kota' ELSE 'Kabupaten' END AS jenis_wilayah,
    COUNT(*)                                  AS jumlah_wilayah,
    ROUND(AVG(ipm), 2)                        AS rata_ipm,
    ROUND(AVG(rata_lama_sekolah), 2)          AS rata_rls,
    ROUND(SUM(total_murid) / SUM(total_guru), 2) AS rasio_murid_guru
FROM fact_indikator_pendidikan
GROUP BY 1
ORDER BY 1
""").show()

uji.close()

┌───────────────┬────────────────┬──────────┬──────────┬──────────────────┐
│ jenis_wilayah │ jumlah_wilayah │ rata_ipm │ rata_rls │ rasio_murid_guru │
│    varchar    │     int64      │  double  │  double  │      double      │
├───────────────┼────────────────┼──────────┼──────────┼──────────────────┤
│ Kabupaten     │             29 │    74.09 │     8.34 │             14.1 │
│ Kota          │              9 │    82.42 │    10.88 │            17.03 │
└───────────────┴────────────────┴──────────┴──────────┴──────────────────┘



**(Opsional) DuckLake terkelola di MotherDuck**

MotherDuck juga dapat menyimpan data dalam format DuckLake. Sel berikut membuat database bertipe DuckLake di cloud. Jika akun/versi DuckDB belum mendukung, sel akan menampilkan pesan dan bisa dilewati; dashboard tetap memakai `jatim_dw`.

In [ ]:
try:
    md_con = duckdb.connect("md:")
    md_con.execute("CREATE DATABASE IF NOT EXISTS jatim_lake (TYPE DUCKLAKE)")
    md_con.close()

    con.execute("ATTACH 'md:jatim_lake'")

    for tabel in tabel_publikasi:
        con.execute(f"""
        CREATE OR REPLACE TABLE jatim_lake.main.{tabel} AS
        SELECT * FROM lake_jatim.{tabel}
        """)

    con.sql("SELECT COUNT(*) AS jumlah_data FROM jatim_lake.main.fact_indikator_pendidikan").show()
    print("DuckLake di MotherDuck berhasil dibuat.")

except Exception as e:
    print("DuckLake di MotherDuck dilewati:", e)

┌─────────────┐
│ jumlah_data │
│    int64    │
├─────────────┤
│          38 │
└─────────────┘

DuckLake di MotherDuck berhasil dibuat.


In [ ]:
# Ekspor cadangan CSV untuk mode lokal dashboard (dipakai jika token MotherDuck tidak tersedia)
os.makedirs("/content/dashboard/data", exist_ok=True)

con.sql("""
SELECT
    kabupaten_kota, tahun, ipm,
    sekolah_sd, sekolah_smp, sekolah_sma,
    guru_sd, guru_smp, guru_sma,
    murid_sd, murid_smp, murid_sma,
    rata_lama_sekolah
FROM jatim_dw.main.fact_indikator_pendidikan
ORDER BY kabupaten_kota
""").df().to_csv("/content/dashboard/data/indikator_jatim.csv", index=False)

print("CSV cadangan tersimpan di /content/dashboard/data/indikator_jatim.csv")

CSV cadangan tersimpan di /content/dashboard/data/indikator_jatim.csv


In [ ]:
con.close()
print("Koneksi ditutup. Data warehouse siap dibaca oleh dashboard Streamlit.")

Koneksi ditutup. Data warehouse siap dibaca oleh dashboard Streamlit.


**Pembuatan Dashboard Streamlit**

Dashboard tidak dijalankan di Colab, tetapi di repositori terpisah (`app.py`, `requirements.txt`, `data/indikator_jatim.csv`).

1. Unggah folder `dashboard/` ke repositori GitHub (public).
2. Buka https://share.streamlit.io → *Create app* → pilih repositori, *Main file path* = `app.py`.
3. *Advanced settings → Secrets*, isi:
   ```
   MOTHERDUCK_TOKEN = "token_anda"
   ```
4. *Deploy*. Sidebar dashboard menampilkan **Sumber data: MotherDuck** jika koneksi berhasil.

Uji lokal: `pip install -r requirements.txt` lalu `streamlit run app.py`.